## Read data

In [ ]:
import numpy as np
import pandas as pd
npz_path = './datasets/flow_features.npz'
data = np.load(npz_path)
X = data['X'].astype(np.float32)
y = data['y'].astype(np.int32)
print('Loaded npz:', npz_path)
print('X shape:', X.shape)
print('y binary distribution:', np.unique(y, return_counts=True))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import confusion_matrix, roc_auc_score, f1_score
from sklearn.model_selection import train_test_split

seed = 42
np.random.seed(seed)
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        X_sample = self.X[idx]  

        X_seq = X_sample

        X_hand = X_sample.reshape(-1)  

        label = self.y[idx]
        return X_seq, X_hand, label


def calculate_metrics(y_true, y_pred, y_proba):
    y_true = y_true.cpu().numpy()
    y_pred = y_pred.cpu().numpy()
    y_proba = y_proba.cpu().numpy()

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        'accuracy': (tp + tn) / (tp + tn + fp + fn),
        'precision': tp / (tp + fp + 1e-7),
        'recall': tp / (tp + fn + 1e-7),
        'f1': f1_score(y_true, y_pred),
    }

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=seed, stratify=y
)

train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## Generate Flows Features

In [ ]:
class CNN_BiLSTMAttention(nn.Module):
    def __init__(self, input_dim=50, hidden_size=64, num_classes=1):
        super(CNN_BiLSTMAttention, self).__init__()
        self.hidden_size = hidden_size

        # CNN 
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=128, kernel_size=3, padding=1)  
        self.bn1 = nn.BatchNorm1d(128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

        # BiLSTM
        self.lstm = nn.LSTM(input_size=128, hidden_size=hidden_size,
                            num_layers=1, bidirectional=True, batch_first=True)

        self.attention = nn.Sequential(
            nn.Linear(2 * hidden_size, 1, bias=False)
        )

        self.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(2 * hidden_size, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x, return_features=False, return_attention=False):
        # CNN
        x = x.transpose(1, 2)  
        cnn_out = self.conv1(x)
        cnn_out = self.bn1(cnn_out)
        cnn_out = self.relu(cnn_out)
        cnn_out = self.dropout(cnn_out)
        cnn_out = cnn_out.transpose(1, 2)
 
        # LSTM
        lstm_out, _ = self.lstm(cnn_out)

        # Mask
        mask = (cnn_out.abs().sum(dim=-1) != 0).unsqueeze(-1)

        attn_scores = self.attention(lstm_out)  
        attn_scores = attn_scores.masked_fill(~mask, -1e9)  
        attn_weights = torch.softmax(attn_scores, dim=1)    

        context = torch.sum(lstm_out * attn_weights, dim=1)  

        if return_attention:
            return context, attn_weights.squeeze(-1)  
        elif return_features:
            return context
        else:
            output = self.fc(context)
            return output

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') 
model = CNN_BiLSTMAttention(input_dim=6).to(device)
print(sum(p.numel() for p in model.parameters()))
model.eval()

def get_param_count(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

param_count = get_param_count(model)
print(f"Model Parameter Count:{param_count}")


In [ ]:
import time
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 30

start_time = time.time()
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []

    
    for batch_X, _, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()

        batch_y = batch_y.view(-1, 1)
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)

        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()
        
        all_preds.append(preds.detach())
        all_labels.append(batch_y.detach())
        all_probs.append(probs.detach())
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")
    
    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    all_probs = torch.cat(all_probs)

    model.eval()
    val_preds, val_labels, val_probs = [], [], []
    with torch.no_grad():
        for batch_X,_, batch_y in test_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X).squeeze()
                
                probs = torch.sigmoid(outputs)
                preds = (probs > 0.5).float()

                val_preds.append(preds)
                val_labels.append(batch_y)
                val_probs.append(probs)

end_time = time.time()

print(f"Training elapsed time: {end_time - start_time:.2f} seconds")
torch.save(model.state_dict(), "./models/cnn_bilstm.pth")

In [ ]:
import joblib
import time
def extract_features(model, dataloader, device):
    features_list = []
    labels_list = []

    model.eval()
    with torch.no_grad():
        for batch_X_seq, batch_X_hand, batch_y in dataloader:
            # -------- Move to GPU --------
            batch_X_seq = batch_X_seq.to(device)      # (B, 20, 6)
            batch_X_hand = batch_X_hand.to(device)    # (B, 120)
            batch_y = batch_y.to(device)

            feature_vectors = model(batch_X_seq, return_features=True)
            fused_features = torch.cat([feature_vectors, batch_X_hand], dim=1)

            features_list.append(fused_features.cpu())
            labels_list.append(batch_y.cpu())

    features = torch.cat(features_list, dim=0).numpy()
    labels = torch.cat(labels_list, dim=0).numpy().astype(np.int64)

    return features, labels

# Extract features from the training/test datasets
xgb_train_features, xgb_train_labels = extract_features(model, train_loader, device)
start_time = time.time()
xgb_test_features, xgb_test_labels = extract_features(model, test_loader, device)
end_time = time.time()

print(f'Feature extraction elapsed time: {end_time - start_time:.2f} seconds')
print(f'Training feature shape: {xgb_train_features.shape}, labels: {xgb_train_labels.shape}')
print(f'Test feature shape: {xgb_test_features.shape}, labels: {xgb_test_labels.shape}')


In [ ]:
def calculate_metrics(y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "accuracy": (tp + tn) / (tp + tn + fp + fn),
        "precision": tp / (tp + fp + 1e-7),
        "recall": tp / (tp + fn + 1e-7),
        "f1": f1_score(y_true, y_pred),
        "auc": roc_auc_score(y_true, y_proba),
        "specificity": tn / (tn + fp + 1e-7)
    }

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

xgb_model = xgb.XGBClassifier(n_estimators=200, learning_rate=0.2, max_depth=6, subsample=1.0, random_state=42)
xgb_model.fit(xgb_train_features, xgb_train_labels)

# Prediction and evaluation
y_pred = xgb_model.predict(xgb_test_features)
y_true = pd.Series(xgb_test_labels.astype(np.int64))
y_pred = pd.Series(y_pred)

train_preds = xgb_model.predict(xgb_train_features)
test_preds = xgb_model.predict(xgb_test_features)

train_probs = xgb_model.predict_proba(xgb_train_features)[:, 1]
test_probs = xgb_model.predict_proba(xgb_test_features)[:, 1]

# Evaluation function
def evaluate_model(y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_proba)
    report = classification_report(y_true, y_pred, digits=4)
    cm = confusion_matrix(y_true, y_pred)
    return acc, auc, report, cm

# Training set evaluation
train_acc, train_auc, train_report, train_cm = evaluate_model(xgb_train_labels, train_preds, train_probs)
print(f'Training accuracy: {train_acc:.4f}, AUC: {train_auc:.4f}')
print('Training classification report:', train_report)
print('Training confusion matrix:', train_cm)
print(calculate_metrics(torch.tensor(xgb_train_labels.astype(np.int64)), torch.tensor(train_preds), torch.tensor(train_probs)))

# Test set evaluation
test_acc, test_auc, test_report, test_cm = evaluate_model(xgb_test_labels, test_preds, test_probs)
print(f'Test accuracy: {test_acc:.4f}, AUC: {test_auc:.4f}')
print('Test classification report:', test_report)
print('Test confusion matrix:', test_cm)
print(calculate_metrics(torch.tensor(xgb_test_labels.astype(np.int64)), torch.tensor(test_preds), torch.tensor(test_probs)))
